### FAISS (Facebook AI Similarity Search)

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader = TextLoader("speech.txt")
docs = loader.load()

text_splitter = CharacterTextSplitter(chunk_size = 1000, chunk_overlap = 30)
final_docs = text_splitter.split_documents(docs)
final_docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct our

In [8]:
embedding_model = OllamaEmbeddings(model="gemma2:2b")

db = FAISS.from_documents(final_docs, embedding_model)
db

In [10]:
### Querying
query = "How does the speaker describe the desired outcome of the war?"
docs = db.similarity_search(query)
docs[0].page_content


'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

#### As a Retriever
We can also convert the VectorStore into a Retriever Class. This allows us to work closed with LangChain methods, which largely works with retrievers.

In [12]:
retriever = db.as_retriever()
docs = retriever.invoke(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

#### Similarity Search with score 
In LangChain's FAISS vector store, similarity_search_with_score() returns both:

1. The retrieved document
2. Its similarity/distance score

##### Understanding the Score

With FAISS, the score is usually a distance, not a similarity.
That means:

Lower Score = More Similar
Higher Score = Less Similar

In [13]:
docs_and_score = db.similarity_search_with_score(query)
docs_and_score

[(Document(id='b3483414-c26a-4b54-bfc2-a484685cc228', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  np.float32(8021.116)),
 (Document(id='198b130f-7775-4ca5-9d3d-6aff6d5ff567', metadata={'source': 'spee

In [14]:
## we can also search the query as documents in FAISS
embedding_vector = embedding_model.embed_query(query)
embedding_vector


[1.259456753730774,
 -0.09910088777542114,
 -1.5610051155090332,
 -0.9827038049697876,
 -0.7046330571174622,
 -0.2971469461917877,
 0.5917938351631165,
 0.8096166253089905,
 0.11370407044887543,
 -0.22091439366340637,
 -1.201053500175476,
 -0.881571352481842,
 -0.40373149514198303,
 0.1565570831298828,
 -3.4036636352539062,
 -2.246678113937378,
 -0.25390809774398804,
 -0.3755098879337311,
 -2.497738838195801,
 -0.6998061537742615,
 0.5630425810813904,
 -1.8880702257156372,
 -1.8069064617156982,
 1.3881354331970215,
 -2.461223840713501,
 -2.534376382827759,
 1.5498111248016357,
 -0.6217157244682312,
 -2.159379005432129,
 2.41241455078125,
 -3.2453136444091797,
 -1.5410960912704468,
 1.1883877515792847,
 1.0813920497894287,
 1.7464063167572021,
 1.404314398765564,
 -0.36944591999053955,
 0.536892831325531,
 2.9942116737365723,
 -3.482128381729126,
 1.485605239868164,
 0.27961060404777527,
 -1.4681934118270874,
 -2.9048821926116943,
 -3.6328039169311523,
 -2.521892547607422,
 -0.226732626

In [16]:
docs_and_score = db.similarity_search_by_vector(embedding_vector)
docs_and_score

[Document(id='b3483414-c26a-4b54-bfc2-a484685cc228', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='198b130f-7775-4ca5-9d3d-6aff6d5ff567', metadata={'source': 'speech.txt'}, page_content='…\n

#### Save & Load
When you create a FAISS vector store, everything initially lives in memory.
##
db = FAISS.from_documents(
    documents,
    embeddings
)
##
If you close Python, the vector store disappears.
##
To avoid recreating embeddings every time, you save the FAISS index to disk and later load it back.
##
Saving a FAISS Vector Store
##
Example:
##
from langchain_community.vectorstores import FAISS
##
db = FAISS.from_documents(
    documents,
    embeddings
)
##
db.save_local("faiss_index")
##
This creates a folder:
##
faiss_index/
├── index.faiss
└── index.pkl
What Are These Files?
index.faiss
##
Contains:
##
Vector embeddings
FAISS search index
Nearest-neighbor structure
##
This is the actual FAISS database.
##
index.pkl
##
Contains:
##
Document contents
Metadata
Mappings
##
LangChain uses it to reconstruct the full vector store.

In [17]:
db.save_local("faiss_index")

In [20]:
new_db = FAISS.load_local("faiss_index", embedding_model, allow_dangerous_deserialization=True)
docs = new_db.similarity_search(query)
docs

[Document(id='b3483414-c26a-4b54-bfc2-a484685cc228', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our hearts—for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='198b130f-7775-4ca5-9d3d-6aff6d5ff567', metadata={'source': 'speech.txt'}, page_content='…\n